## Data Preprocessing for ML (Unsupervised Model - Clustering)

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option('display.max_columns', None)

In [2]:
# Reading Data (all clients)
df = pd.read_parquet('Clients_LoanLens_clean.parquet')
print(df.shape)

(347069, 38)


## Feature Engineering

In [3]:
# New feature: income per family member
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']
df['INCOME_PER_PERSON'].describe().round(2)

count      347069.00
mean        93711.35
std         98709.77
min          2812.50
25%         49500.00
50%         75000.00
75%        112500.00
max      39000000.00
Name: INCOME_PER_PERSON, dtype: float64

- **New feature:** `INCOME_PER_PERSON` = income / family members. The same income means a different financial situation for a single client and a family of 5.

In [4]:
df.head()

,CODE_GENDER,NAME_FAMILY_STATUS,CNT_CHILDREN,CNT_FAM_MEMBERS,NAME_EDUCATION_TYPE,AMT_INCOME_TOTAL,NAME_INCOME_TYPE,OCCUPATION_TYPE,ORGANIZATION_TYPE,FLAG_OWN_CAR,OWN_CAR_AGE,FLAG_OWN_REALTY,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,REGION_RATING_CLIENT_W_CITY,LIVE_REGION_NOT_WORK_REGION,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,FLAG_DOCUMENT_3,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_8,FLAG_EMP_PHONE,NAME_CONTRACT_STATUS,AMT_APPLICATION,CNT_PAYMENT,AMT_GOODS_PRICE,AMT_DOWN_PAYMENT,NAME_CONTRACT_TYPE,NAME_CLIENT_TYPE,AMT_ANNUITY,AMT_CREDIT,OTHER_DOCUMENTS,HAS_BUREAU_RECORD,TOTAL_BUREAU_REQ,MONTH_EMPLOYED,AGE_YEARS,INCOME_PER_PERSON
SK_ID_CURR,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
100002,M,Single / not married,0,1,Secondary / secondary special,202500.0,Working,Laborers,Business Entity,N,0.0,Y,House / apartment,0.018801,2,0,0.083037,0.262949,0.139376,1,0,0,0,1,Approved,179055.0,24.0,179055.0,0.0,Consumer loans,New,9251.775,179055.0,0,1,1,21,25,202500.0
100003,F,Married,0,2,Higher education,270000.0,State servant,Core staff,School,N,0.0,N,House / apartment,0.003541,1,0,0.311267,0.622246,NaN,1,0,0,0,1,Approved,900000.0,12.0,900000.0,0.0,Cash loans,Repeater,98356.995,1035882.0,0,1,0,39,45,135000.0
100004,M,Single / not married,0,1,Secondary / secondary special,67500.0,Working,Laborers,Government,Y,26.0,Y,House / apartment,0.010032,2,0,NaN,0.555912,0.729567,0,0,0,0,1,Approved,24282.0,4.0,24282.0,4860.0,Consumer loans,New,5357.250,20106.0,0,1,0,7,52,67500.0
100006,F,Civil marriage,0,2,Secondary / secondary special,135000.0,Working,Laborers,Business Entity,N,0.0,Y,House / apartment,0.008019,2,0,NaN,0.650442,NaN,1,0,0,0,1,Approved,675000.0,48.0,675000.0,0.0,Cash loans,Repeater,24246.000,675000.0,0,0,0,101,52,67500.0
100007,M,Single / not married,0,1,Secondary / secondary special,121500.0,Working,Core staff,Religion,N,0.0,Y,House / apartment,0.028663,2,0,NaN,0.322738,NaN,0,0,0,1,1,Approved,247500.0,24.0,247500.0,0.0,Cash loans,Repeater,16037.640,274288.5,0,1,0,101,54,121500.0


### Data Preprocessing for ML

In [5]:
scal_cols = ['AGE_YEARS', 'REGION_RATING_CLIENT_W_CITY', 'EXT_SOURCE_2']

log_cols = ['INCOME_PER_PERSON']

ohe_cols = ['FLAG_OWN_CAR']

ord_cols = ['NAME_EDUCATION_TYPE']

x = df[scal_cols + log_cols + ohe_cols + ord_cols]

print(x.shape)

(347069, 6)


- **Clustering columns:** income per person (log), age, region rating, credit score (`EXT_SOURCE_2`), car ownership and education.
- First try also included `OWN_CAR_AGE` and `FLAG_OWN_REALTY`: car age created a cluster of its own (old cars, no business meaning) and owning realty was ~68% in every cluster (no separation) → both removed. Education was scaled so it does not dominate.

In [6]:
x.isna().sum()

AGE_YEARS                        0
REGION_RATING_CLIENT_W_CITY      0
EXT_SOURCE_2                   662
INCOME_PER_PERSON                0
FLAG_OWN_CAR                     0
NAME_EDUCATION_TYPE              0
dtype: int64

In [7]:
print(x[scal_cols + log_cols].skew().round(2))

AGE_YEARS                        0.12
REGION_RATING_CLIENT_W_CITY      0.05
EXT_SOURCE_2                    -0.79
INCOME_PER_PERSON              180.22
dtype: float64


- Numerical Columns

In [8]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_pipeline = Pipeline([
    ('Imputer', SimpleImputer(strategy='median')), 
    ('Scaling', StandardScaler())
])
num_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('Imputer', ...), ('Scaling', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account f

In [9]:
from sklearn.preprocessing import FunctionTransformer

log_pipeline = Pipeline([
    ('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('scaler', StandardScaler())
])
log_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('log', ...), ('scaler', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"func func: callable, default=NoneThe callable to use for the transformation. This will be passedthe same arguments as transform, with args and kwargs forwarded.If func is None, then func will be the identity function.",<ufunc 'log1p'>
,"feature_names_out feature_names_out: callable, 'one-to-one' or None, default=NoneDetermines the list of feature names that will be returned by the`get_feature_names_out` method. If it is 'one-to-one', then the outputfeature names will be equal to the input feature names. If it is acallable, then it must take two positional arguments: this`FunctionTransformer` (`self`) and an array-like of input feature names(`input_features`). It must return an array-like of output featurenames. The `get_feature_names_out` method is only defined if`feature_names_out` is not None.See ``get_feature_names_out`` for more details... versionadded:: 1.1",'one-to-one'
,"inverse_func inverse_func: callable, default=NoneThe callable to use for the inverse transformation. This will bepassed the same arguments as inverse transform, with args andkwargs forwarded. If inverse_func is None, then inverse_funcwill be the identity function.",None
,"validate validate: bool, default=FalseIndicate that the input X array should be checked before calling``func``. The possibilities are:- If False, there is no input validation.- If True, then X will be converted to a 2-dimensional NumPy array or sparse matrix. If the conversion is not possible an exception is raised... versionchanged:: 0.22 The default of ``validate`` changed from True to False.",False
,"accept_sparse accept_sparse: bool, default=FalseIndicate that func accepts a sparse matrix as input. If validate isFalse, this has no effect. Otherwise, if accept_sparse is false,sparse matrix inputs will cause an exception to be raised.",False
,"check_inverse check_inverse: bool, default=TrueWhether to check that or ``func`` followed by ``inverse_func`` leads tothe original inputs. It can be used for a sanity check, raising awarning when the condition is not fulfilled... versionadded:: 0.20",True
,"kw_args kw_args: dict, default=NoneDictionary of additional k

- Categorical Columns 

In [10]:
from sklearn.preprocessing import OneHotEncoder

ohe_pipeline = Pipeline(steps=[
    ('OHE', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])
ohe_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('OHE', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",'first'
,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category

In [11]:
from sklearn.preprocessing import OrdinalEncoder

edu_order = ['Lower secondary', 'Secondary / secondary special', 'Incomplete higher',
             'Higher education', 'Academic degree']

ord_pipeline = Pipeline(steps=[
    ('Ord', OrdinalEncoder(categories=[edu_order])),
    ('Scaling', StandardScaler())
])
ord_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('Ord', ...), ('Scaling', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute.","[['Lower secondary', 'Secondary / secondary special', ...]]"
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"handle_unknown handle_unknown: {'error', 'use_encoded_value'}, default='error'When set to 'error' an error will be raised in case an unknowncategorical feature is present during transform. When set to'use_encoded_value', the encoded value of unknown categories will beset to the value given for the parameter `unknown_value`. In:meth:`inverse_transform`, an unknown category will be denoted as None... versionadded:: 0.24",'error'
,"unknown_value unknown_value: int or np.nan, default=NoneWhen the parameter handle_unknown is set to 'use_encoded_value', thisparameter is required and will set the encoded value of unknowncategories. It has to be distinct from the values used to encode any ofthe categories in `fit`. If set to np.nan, the `dtype` parameter mustbe a float dtype... versionadded:: 0.24",None
,"encoded_missing_value encoded_missing_value: int or np.nan, default=np.nanEncoded value of missing categories. If set to `np.nan`, then the `dtype`parameter must be a float dtype... versionadded:: 1.1",nan
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.3 Read more in the :ref:`User Guide <encoder_infrequent_categories>`.",None
,"max_categories max_categories: int, default=NoneSpecifies an upper limit to the number of outp

In [12]:
from sklearn.compose import ColumnTransformer

preprocessing = ColumnTransformer(transformers=[
    ('Num Pipeline', num_pipeline, scal_cols),    
    ('Log Pipeline', log_pipeline, log_cols),
    ('OHE Pipeline', ohe_pipeline, ohe_cols),
    ('Ordinal Pipeline', ord_pipeline, ord_cols)
])
preprocessing

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('Num Pipeline', ...), ('Log Pipeline', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name

In [13]:
x_prep = preprocessing.fit_transform(x)
print(x_prep.shape)
print(np.isnan(x_prep).sum())

(347069, 6)
0


## Cluster Model

- Elbow Method

In [14]:
from sklearn.cluster import KMeans

k_values = range(2, 11)

inertia_list_scores = []

for k in k_values:

    kmeans = KMeans(n_clusters= k, random_state= 42)

    labels = kmeans.fit_predict(x_prep)

    inertia_score = round(kmeans.inertia_, 2)

    inertia_list_scores.append(inertia_score)

print(inertia_list_scores)

[1488561.39, 1220206.85, 1089107.49, 1009663.47, 940494.45, 903314.92, 813388.21, 767489.43, 711033.87]


In [15]:
px.line(x= k_values, y= inertia_list_scores, markers= True,
        labels= {'x' : 'Clusters', 'y' : 'Inertia Score'})

- Silhouette Score

In [16]:
from sklearn.metrics import silhouette_score

k_values = range(2, 11)

silhouette_score_list = []

for k in k_values:

    kmeans = KMeans(n_clusters= k, random_state= 42)

    labels = kmeans.fit_predict(x_prep)

    # sample_size: silhouette on 347K rows needs huge memory → random sample of 20K
    score = round(silhouette_score(x_prep, labels, sample_size= 20000, random_state= 42), 2)

    silhouette_score_list.append(score)

print(silhouette_score_list)

[0.17, 0.21, 0.18, 0.18, 0.21, 0.21, 0.2, 0.2, 0.2]


In [17]:
px.line(x= k_values, y= silhouette_score_list, markers= True,
        labels= {'x' : 'Clusters', 'y' : 'Silhouette Score'})

- **Silhouette is low (0.17–0.21)** for all k: clients are not naturally separated into groups, their income, age and credit score change gradually. So k is chosen by the Elbow (4–5), balanced cluster sizes and business meaning.

In [18]:
for k in [4, 5]:
    kmeans = KMeans(n_clusters= k, random_state= 42)
    labels = kmeans.fit_predict(x_prep)
    print(f'k = {k}')
    print((pd.Series(labels).value_counts(normalize= True).sort_index() * 100).round(1))
    print('-' * 30)

k = 4
0    21.8
1    28.2
2    24.2
3    25.8
Name: proportion, dtype: float64
------------------------------
k = 5
0    15.7
1    23.3
2    23.1
3    15.0
4    22.8
Name: proportion, dtype: float64
------------------------------


In [19]:
for k in [4, 5]:
    kmeans = KMeans(n_clusters= k, random_state= 42)
    df[f'Cluster_{k}'] = kmeans.fit_predict(x_prep)

    profile = df.groupby(f'Cluster_{k}').agg(
        Clients       = ('AGE_YEARS', 'size'),
        Income_Person = ('INCOME_PER_PERSON', 'median'),
        Age           = ('AGE_YEARS', 'mean'),
        Credit_Score  = ('EXT_SOURCE_2', 'mean'),
        Region_Rating = ('REGION_RATING_CLIENT_W_CITY', 'mean'),
        Car_Age       = ('OWN_CAR_AGE', 'mean'),
        Has_Car       = ('FLAG_OWN_CAR', lambda s: (s == 'Y').mean() * 100),
        Owns_Realty   = ('FLAG_OWN_REALTY', lambda s: (s == 'Y').mean() * 100),
        Higher_Edu    = ('NAME_EDUCATION_TYPE', lambda s: s.isin(['Higher education', 'Academic degree']).mean() * 100)
    ).round(2)

    print(f'k = {k}')
    display(profile)

k = 4


,Clients,Income_Person,Age,Credit_Score,Region_Rating,Car_Age,Has_Car,Owns_Realty,Higher_Edu
Cluster_4,,,,,,,,,
0,75678,60000.0,42.64,0.25,2.40,4.08,29.07,69.44,7.01
1,97905,90000.0,54.26,0.60,1.91,3.12,26.60,74.30,0.00
2,83845,94500.0,40.73,0.58,1.92,4.43,42.50,67.94,95.29
3,89641,56250.0,34.76,0.58,1.95,4.85,38.42,65.35,0.04


k = 5


,Clients,Income_Person,Age,Credit_Score,Region_Rating,Car_Age,Has_Car,Owns_Realty,Higher_Edu
Cluster_5,,,,,,,,,
0,54630,67500.0,56.31,0.37,2.36,3.04,21.59,75.16,3.73
1,80991,112500.0,50.82,0.63,1.77,3.47,30.90,72.73,0.00
2,80196,96000.0,40.81,0.59,1.90,4.44,42.81,68.00,95.17
3,52146,67500.0,33.02,0.24,2.29,4.72,34.46,65.07,12.65
4,79106,49500.0,36.44,0.59,2.03,4.71,36.66,66.25,0.36


In [20]:
for k in [4, 5]:
    print(f'k = {k}')
    print((df.groupby(f'Cluster_{k}')['NAME_CONTRACT_STATUS']
             .apply(lambda s: (s == 'Refused').sum() / s.notna().sum() * 100)
             .round(1)))
    print('-' * 30)

k = 4
Cluster_4
0    16.1
1    16.9
2    14.6
3    13.7
Name: NAME_CONTRACT_STATUS, dtype: float64
------------------------------
k = 5
Cluster_5
0    16.3
1    17.2
2    14.6
3    15.7
4    13.4
Name: NAME_CONTRACT_STATUS, dtype: float64
------------------------------


- **k = 5 selected:** balanced sizes (15–23%) and clearer business segments (it separates the most affluent clients and splits high-risk clients by age).

## Final Model

In [21]:
# Final model: preprocessing + KMeans (k = 5) in one pipeline (for the Streamlit app)
kmeans_pipeline = Pipeline(steps=[
    ('Preprocessing', preprocessing),
    ('Model', KMeans(n_clusters= 5, random_state= 42))
])

df['Segment'] = kmeans_pipeline.fit_predict(x)

df.drop(columns=['Cluster_4', 'Cluster_5'], inplace= True)

(df['Segment'].value_counts(normalize= True).sort_index() * 100).round(1)

Segment
0    15.7
1    23.3
2    23.1
3    15.0
4    22.8
Name: proportion, dtype: float64

In [22]:
mapping_dict = {
    0 : 'Subprime – Mature',
    1 : 'Prime – Mass Affluent',
    2 : 'Prime – Professionals',
    3 : 'Subprime – Young',
    4 : 'Near-Prime – Emerging'
}

df['Segment_Name'] = df['Segment'].map(mapping_dict)
df['Segment_Name'].value_counts()

Segment_Name
Prime – Mass Affluent    80991
Prime – Professionals    80196
Near-Prime – Emerging    79106
Subprime – Mature        54630
Subprime – Young         52146
Name: count, dtype: int64

| Segment | Name | Profile |
|---|---|---|
| 0 | Subprime – Mature | Older (56), weak credit score (0.37), lower-rated regions |
| 1 | Prime – Mass Affluent | Highest income per person (112.5K), age 51, best credit score (0.63) |
| 2 | Prime – Professionals | 95% higher education, high income (96K), good credit score (0.59) |
| 3 | Subprime – Young | Young (33), lowest credit score (0.24) |
| 4 | Near-Prime – Emerging | Young (36), lowest income (49.5K) but good credit score (0.59) |

- **Credit quality:** Prime / Near-Prime / Subprime. **Income tier:** Mass Affluent / Emerging.
- Refusal rate is similar across segments (13–17%): refusal depends mainly on the application (contract type, client type), not the client profile. The segmentation describes **who the client is**, while the approval model judges **the application**.

In [23]:
# Sample of 5,000 clients (plotting 347K points makes the notebook very heavy)
df_sample = df.sample(5000, random_state= 42)

px.scatter(data_frame= df_sample, x= 'EXT_SOURCE_2', y= 'INCOME_PER_PERSON', color= 'Segment_Name',
           log_y= True, opacity= 0.6,
           labels= {'EXT_SOURCE_2' : 'Credit Score', 'INCOME_PER_PERSON' : 'Income per Person (log)'},
           title= 'Client Segments: Credit Score vs Income per Person')

In [24]:
px.box(data_frame= df_sample, x= 'Segment_Name', y= 'AGE_YEARS', color= 'Segment_Name',
       labels= {'AGE_YEARS' : 'Age', 'Segment_Name' : 'Segment'},
       title= 'Age by Segment')

- **Scatter:** Subprime segments sit on the left (credit score < 0.45). Mass Affluent is top-right (high income and credit), Near-Prime – Emerging is bottom-right (good credit, lower income). Professionals overlap here because they are separated by education, which is not on these axes.
- **Box plot:** the two Subprime segments overlap in the scatter but are clearly separated by age (Young ≈ 32 vs Mature ≈ 57).

- Agglomerative

In [25]:
from sklearn.cluster import AgglomerativeClustering

# Agglomerative needs a full distance matrix → sample of 10,000 clients
x_sample = x_prep[np.random.RandomState(42).choice(len(x_prep), 10000, replace= False)]

k_values = range(2, 11)

silhouette_score_list_agg = []

for k in k_values:

    agglomorative = AgglomerativeClustering(n_clusters= k)

    labels = agglomorative.fit_predict(x_sample)

    score = round(silhouette_score(x_sample, labels), 2)

    silhouette_score_list_agg.append(score)

print(silhouette_score_list_agg)

[0.22, 0.19, 0.2, 0.21, 0.16, 0.15, 0.16, 0.16, 0.17]


- **Agglomerative (sample of 10,000):** silhouette 0.15–0.22, the same range as KMeans → both confirm clients are not naturally separated.
- **KMeans selected:** it scales to all 347K clients and has `predict()` to assign a **new client** to a segment in the app. Agglomerative needs a full distance matrix (sample only) and cannot predict new clients.

- Save Model

In [26]:
import joblib

# Profile table of the final segments (shown in the app with each client's segment)
segment_profile = df.groupby('Segment_Name').agg(
    Clients       = ('AGE_YEARS', 'size'),
    Income_Person = ('INCOME_PER_PERSON', 'median'),
    Age           = ('AGE_YEARS', 'mean'),
    Credit_Score  = ('EXT_SOURCE_2', 'mean'),
    Has_Car       = ('FLAG_OWN_CAR', lambda s: (s == 'Y').mean() * 100),
    Owns_Realty   = ('FLAG_OWN_REALTY', lambda s: (s == 'Y').mean() * 100),
    Higher_Edu    = ('NAME_EDUCATION_TYPE', lambda s: s.isin(['Higher education', 'Academic degree']).mean() * 100)
).round(2)

segment_profile

,Clients,Income_Person,Age,Credit_Score,Has_Car,Owns_Realty,Higher_Edu
Segment_Name,,,,,,,
Near-Prime – Emerging,79106,49500.0,36.44,0.59,36.66,66.25,0.36
Prime – Mass Affluent,80991,112500.0,50.82,0.63,30.90,72.73,0.00
Prime – Professionals,80196,96000.0,40.81,0.59,42.81,68.00,95.17
Subprime – Mature,54630,67500.0,56.31,0.37,21.59,75.16,3.73
Subprime – Young,52146,67500.0,33.02,0.24,34.46,65.07,12.65


In [27]:
# Save model, segment names, input columns and profile (for the Streamlit app)
joblib.dump(kmeans_pipeline, 'KMeans_Segment_Pipeline.pkl')
joblib.dump(mapping_dict, 'Segment_Names.pkl')
joblib.dump(x.columns.tolist(), 'Segment_Inputs.pkl')
segment_profile.to_csv('Segment_Profile.csv')

- Test Model

In [28]:
model = joblib.load('KMeans_Segment_Pipeline.pkl')
names = joblib.load('Segment_Names.pkl')

sample = x.sample(5, random_state= 42)
pred = model.predict(sample)

print([names[p] for p in pred])
print(df.loc[sample.index, 'Segment_Name'].tolist())   # actual

['Prime – Professionals', 'Near-Prime – Emerging', 'Prime – Professionals', 'Prime – Mass Affluent', 'Subprime – Young']
['Prime – Professionals', 'Near-Prime – Emerging', 'Prime – Professionals', 'Prime – Mass Affluent', 'Subprime – Young']


In [3]:
! pip list | findstr /i "streamlit pandas numpy pyarrow plotly scikit-learn xgboost category joblib fpdf"

category_encoders         2.10.0
fpdf2                     2.9.0
joblib                    1.5.3
numpy                     2.4.4
pandas                    3.0.3
plotly                    6.8.0
pyarrow                   24.0.0
scikit-learn              1.9.0
streamlit                 1.61.1
xgboost                   3.2.0
